In [1]:
# imports
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from transformers import pipeline
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
from edgar import Company, set_identity
import regex as re
import torch
import time

DATA_DIR = '../../../data'            # shared raw inputs and caches
OUT_DIR = '../../../outputs/v3/data'  # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)

### Gathering Data

In [2]:
# Universe: S&P 500. The list is saved on the first run and reused afterwards,
# so reruns use the same tickers the EODHD news was downloaded for.
UNIVERSE_PATH = f'{OUT_DIR}/sp500_tickers.csv'
if os.path.exists(UNIVERSE_PATH):
    table = pd.read_csv(UNIVERSE_PATH)
else:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    html = requests.get(url, headers=headers).text
    table = pd.read_html(html)[0]
    table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
    table = table[['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']]
    table.to_csv(UNIVERSE_PATH, index=False)
table.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


#### Tickers
The universe is the S&P 500 (current constituents, saved to `sp500_tickers.csv`).

The original 19 AI-sector tickers are kept in `ai_tickers` for subsample comparisons:<br>
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom),
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalers**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [36]:
# list of all the tickers with a dictionary that sorts them by type of stock
tickers = pd.read_csv(f'{DATA_DIR}/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
ai_tickers_flat = [t for group in ai_tickers.values() for t in group]
all_tickers = table['Symbol'].tolist()
print(f'{len(all_tickers)} tickers')

503 tickers


In [37]:
# Data from 2018 to today: 2019-2025 is the development sample, 2026 the holdout
prices = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start='2018-01-01',
    end=(datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=False
)
prices.to_csv(f'{OUT_DIR}/prices_raw.csv')

In [38]:
prices.head()

Price           Close                                                   \
Ticker              A       AAPL       ABBV ABNB        ABT       ACGL   
Date                                                                     
2018-01-02  63.306431  40.201832  67.870338  NaN  49.841591  27.988113   
2018-01-03  64.917183  40.194839  68.932426  NaN  49.951794  28.013470   
2018-01-04  64.430229  40.381531  68.539337  NaN  49.867027  28.118069   
2018-01-05  65.460342  40.841286  69.732460  NaN  50.011139  28.007132   
2018-01-08  65.600838  40.689590  68.615196  NaN  49.867027  28.010302   

Price                                                     ...   Volume  \
Ticker             ACN        ADBE        ADI        ADM  ...       WY   
Date                                                      ...            
2018-01-02  132.881607  177.699997  77.155228  31.005114  ...  3144000   
2018-01-03  133.494888  181.039993  78.112427  30.765358  ...  2922800   
2018-01-04  135.075592  183.220001  78.026962  31.283548  ...  2727800   
2018-01-05  136.189850  185.339996  78.343163  31.074732  ...  2924800   
2018-01-08  137.278198  185.039993  78.479897  31.005114  ...  2760800   

Price                                                                       \
Ticker         WYNN      XEL       XOM     XYL       XYZ      YUM      ZBH   
Date                                                                         
2018-01-02  2581200  2443400  11469300  877800  11240500  1747800  1818259   
2018-01-03  3530900  4114900  13957700  778800  11617900  2554900  1368664   
2018-01-04  2708800  2807000  10863000  796500  20469800  1971200  1105396   
2018-01-05  2186600  3728000  11047600  666800  31102000  1927100  1095302   
2018-01-08  1803800  3837600  10927100  554700  26626100  1599400  1286985   

Price                        
Ticker        ZBRA      ZTS  
Date                         
2018-01-02  310600  2135600  
2018-01-03  253000  2328200  
2018-01-04  435200  2534000  
2018-01-05  301800  2166100  
2018-01-08  218600  3631400  

[5 rows x 2525 columns]

### Cleaning Data

In [47]:
close  = prices['Close']
volume = prices['Volume']
soxx   = close['SOXX']
qqq    = close['QQQ']

In [48]:
# Remove tickers with more than 5% NaN values
clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices.to_csv(f'{OUT_DIR}/clean_prices.csv')

In [49]:
# Convert prices to log
log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()

### Feature Engineering

In [5]:
import sys
sys.path.insert(0, '../../../src')
sys.path.insert(0, '../../../scripts')
from features import make_features
from score_sentiment import assign_trading_date

In [51]:
all_feature_dfs = []
for ticker in all_tickers:
    if ticker not in close.columns:
        continue
    ticker_feats = make_features(
        close  = close[ticker],
        volume = volume[ticker],
        soxx   = soxx,
        qqq    = qqq
    )
    ticker_feats['Ticker'] = ticker      
    all_feature_dfs.append(ticker_feats)

In [52]:
features_df = pd.concat(all_feature_dfs).reset_index()
features_df = features_df.set_index(['Date','Ticker'])

In [53]:
# Forward 5-day cumulative return
features_df['fwd_5d'] = (
    features_df.groupby('Ticker')['ret_1d']
    .transform(lambda x: x.rolling(5).sum().shift(-5))
)

features_df = features_df.dropna(subset=['fwd_5d'])

# Cross-sectional: top half = 1, bottom half = 0 (guaranteed 50/50 per date)
# Which stocks will outperform their peers? 
features_df['Target'] = (
    features_df.groupby('Date')['fwd_5d']
    .transform(lambda x: (x > x.median()).astype(int))
)

In [54]:
features_df.head()

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,fwd_5d,Target
Date,Ticker,,,,,,,,,,,,,,
2019-01-02,MMM,0.002152,0.001311,0.069029,-0.009698,-0.081618,0.019782,0.019320,43.774078,-0.022212,0.001318,0.920058,-0.242101,0.008613,0
2019-01-03,MMM,-0.037653,-0.029624,-0.013421,-0.055509,-0.119754,0.022619,0.020605,35.231612,0.017181,0.008380,1.228533,-0.270638,0.053026,0
2019-01-04,MMM,0.041140,0.004094,0.003251,0.007160,-0.053808,0.028271,0.022245,41.589681,-0.001133,0.000760,1.089791,-0.240632,0.004706,0
2019-01-07,MMM,-0.002300,-0.000367,0.007974,0.018679,-0.058220,0.028019,0.022215,45.779821,-0.027993,-0.008201,0.798790,-0.242379,0.006798,0
2019-01-08,MMM,0.004191,0.043099,0.005983,0.043157,-0.033091,0.027957,0.021794,49.588174,-0.036040,-0.024394,0.912325,-0.239203,-0.014201,0


In [55]:
features_df.to_csv(f'{OUT_DIR}/features_df.csv')

### News Sentiment (EODHD)
News is scored outside this notebook, because scoring ~500 tickers takes hours. Run these from the repository root, **after** the cell above saves `features_df.csv`:

```
caffeinate -i python scripts/download_news.py > logs/download_news.log 2>&1      # already done
caffeinate -i python scripts/score_sentiment.py > logs/score_sentiment.log 2>&1
```

`score_sentiment.py` assigns each article to the first trading day whose 4pm ET close comes after it was published, and writes `outputs/v3/data/sentiment_df.csv`.

In [3]:
# merge sentiment df back to features df
sentiment_df = pd.read_csv(f'{OUT_DIR}/sentiment_df.csv', parse_dates=['Date'])
news_cols = ['news_sentiment', 'news_count', 'pct_positive', 'eod_sentiment']

df = features_df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.merge(sentiment_df, on=['Date', 'Ticker'], how='left')
df = df.set_index(['Date', 'Ticker'])

# Fill missing sentiment (days with no news)
df[news_cols] = df[news_cols].fillna(0)

### SEC 8K Filings (Earnings Releases)

Filings are downloaded and scored by `scripts/score_filings.py` (zero-shot topic classification with DeBERTa, plus FinBERT tone on AI sentences). Run it from the repository root after `sp500_tickers.csv` exists:

```
python3 scripts/score_filings.py --time-only NVDA MMM                       # estimate run time
caffeinate -i python3 scripts/score_filings.py > logs/score_filings.log 2>&1
```

It saves each filing's text and scores per ticker, so it can be stopped and restarted, and writes `outputs/v3/data/filing_signals_df.csv`.

In [ ]:
# Filing-level signals from scripts/score_filings.py
transcript_signals_df = pd.read_csv(f'{OUT_DIR}/filing_signals_df.csv')
text_cols = [col for col in transcript_signals_df.columns if col not in ('Ticker', 'Date', 'Accepted')]

# Earnings 8-Ks are often filed after the 4pm ET close. Count each filing on the first trading
# day whose close comes after the SEC accepted it (same rule as score_sentiment.py uses for news).
trading_days = features_df.index.get_level_values('Date').unique().sort_values()
transcript_signals_df['Date'] = assign_trading_date(transcript_signals_df['Accepted'], trading_days)
transcript_signals_df = transcript_signals_df.dropna(subset=['Date'])      # filed after the panel ends
transcript_signals_df['FilingDay'] = transcript_signals_df['Date']          # kept for the test

transcript_signals_df = (transcript_signals_df.drop(columns='Accepted')
                         .set_index(['Date', 'Ticker']).sort_index())

transcript_signals_df.to_csv(f'{OUT_DIR}/transcript_signals_df.csv')
transcript_signals_df.head()

ai_demand_up_share  ai_demand_down_share  capex_up_share  \
Date       Ticker                                                             
2019-01-02 A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           A                      0.0                   0.0             0.0   
           AAPL                   0.0                   0.0             0.0   

                   capex_down_share  competition_share  supply_limited_share  \
Date       Ticker                                                              
2019-01-02 A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           A                    0.0                0.0                   0.0   
           AAPL                 0.0                0.0                   0.0   

                   ai_demand_net  capex_net  ai_tone  ai_sentence_ratio  
Date       Ticker                                                        
2019-01-02 A                 0.0        0.0      0.0                0.0  
           A                 0.0        0.0      0.0                0.0  
           A                 0.0        0.0      0.0                0.0  
           A                 0.0        0.0      0.0                0.0  
           AAPL              0.0        0.0      0.0                0.0

In [5]:
# merge back to original dataframe
df = df.reset_index()

df = pd.merge_asof(
    df.sort_values('Date'),
    transcript_signals_df.reset_index().rename(columns={'date': 'Date'}).sort_values('Date'),
    on='Date',
    by='Ticker',
    direction='backward',
    tolerance=pd.Timedelta(days=120) # Old filing scores become 0 after 1 quarter
)

with_domain_keywords = df.set_index(['Date', 'Ticker'])
with_domain_keywords[text_cols] = with_domain_keywords[text_cols].fillna(0)

### Final Cleaning

In [6]:
edgar_cols = text_cols

In [7]:
with_domain_keywords[edgar_cols] = with_domain_keywords[edgar_cols].fillna(0)

In [8]:
with_domain_keywords.to_csv(f'{OUT_DIR}/ai_sector_features_df.csv')

In [9]:
with_domain_keywords.head()

ret_1d    ret_3d    ret_5d   ret_10d   ret_20d    vol_5d  \
Date       Ticker                                                               
2019-01-02 MMM     0.002152  0.001311  0.069029 -0.009698 -0.081618  0.019782   
           COHR   -0.016020 -0.005913  0.079784 -0.027702 -0.146446  0.031812   
           CL     -0.004537 -0.007372  0.020847 -0.058328 -0.067223  0.011292   
           UAL     0.005374  0.013728  0.063953 -0.023547 -0.129473  0.026668   
           CMCSA   0.009398  0.002645  0.045091 -0.045536 -0.114044  0.014587   

                    vol_20d     rsi_14  rel_soxx   rel_qqq  ...  \
Date       Ticker                                           ...   
2019-01-02 MMM     0.019320  43.774078 -0.022212  0.001318  ...   
           COHR    0.028103  42.649455 -0.087040 -0.063510  ...   
           CL      0.014477  25.178504 -0.007817  0.015713  ...   
           UAL     0.025303  42.053255 -0.070067 -0.046537  ...   
           CMCSA   0.014900  31.970932 -0.054638 -0.031108  ...   

                   ai_demand_up_share  ai_demand_down_share  capex_up_share  \
Date       Ticker                                                             
2019-01-02 MMM                    0.0                   0.0        0.000000   
           COHR                   0.0                   0.0        0.000000   
           CL                     0.0                   0.0        0.000000   
           UAL                    0.0                   0.0        0.000000   
           CMCSA                  0.0                   0.0        0.015267   

                   capex_down_share  competition_share  supply_limited_share  \
Date       Ticker                                                              
2019-01-02 MMM                  0.0                0.0                   0.0   
           COHR                 0.0                0.0                   0.0   
           CL                   0.0                0.0                   0.0   
           UAL                  0.0                0.0                   0.0   
           CMCSA                0.0                0.0                   0.0   

                   ai_demand_net  capex_net  ai_tone  ai_sentence_ratio  
Date       Ticker                                                        
2019-01-02 MMM               0.0   0.000000      0.0                0.0  
           COHR              0.0   0.000000      0.0                0.0  
           CL                0.0   0.000000      0.0                0.0  
           UAL               0.0   0.000000      0.0                0.0  
           CMCSA             0.0   0.015267      0.0                0.0  

[5 rows x 28 columns]

In [4]:
missing = set(table['Symbol']) - set(transcript_signals_df.index.get_level_values('Ticker').unique())
print(missing)


{'VYLR', 'ECHO'}
